In [ ]:
!apt-get update -qq
!apt-get install -y strace
#installing and updating packages

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  strace
0 upgraded, 1 newly installed, 0 to remove and 43 not upgraded.
Need to get 567 kB of archives.
After this operation, 2,048 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 strace amd64 5.16-0ubuntu3 [567 kB]
Fetched 567 kB in 1s (793 kB/s)
Selecting previously unselected package strace.
(Reading database ... 121703 files and directories currently installed.)
Preparing to unpack .../strace_5.16-0ubuntu3_amd64.deb ...
Unpacking strace (5.16-0ubuntu3) ...
Setting up strace (5.16-0ubuntu3) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
import os
import subprocess
from pathlib import Path
# setting path to get logs
data_directory = "/bin/"

In [ ]:
import logging
import os
import shlex
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, List

BIN_DIR = Path("./Linux-Malware-Samples")
OUTPUT_DIR = Path("malware_strace_logs")
TIME_LIMIT_SECS = 15

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO,format="%(asctime)s %(levelname)s %(name)s :: %(message)s",handlers=[logging.StreamHandler(),logging.FileHandler(OUTPUT_DIR / "trace_runner.log", mode="w")])
log = logging.getLogger("trace-runner")

@dataclass(frozen=True)
class Target:
    path: Path
    log_path: Path

def list_directories(root: Path) -> Iterable[Path]:
    for p in root.iterdir():
        if p.is_file():
            yield p

def is_elf_executable(p: Path) -> bool:
    try:
        res = subprocess.run(["file", str(p)],capture_output=True,text=True,check=True,)
        desc = res.stdout.lower()
        return ("elf" in desc) and ("executable" in desc)
    except subprocess.CalledProcessError:
        return False

def build_targets(paths: Iterable[Path]) -> List[Target]:
    items: List[Target] = []
    for p in paths:
        if is_elf_executable(p):
            items.append(Target(path=p, log_path=OUTPUT_DIR / f"{p.name}.strace.log"))
    return items

def run_strace(t: Target) -> None:
    cmd = ["strace", "-f", "-ttt", "-o", str(t.log_path), str(t.path)]
    try:
        logging.getLogger("trace").info("exec %s", shlex.join(cmd))
        subprocess.run(cmd,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL,timeout=TIME_LIMIT_SECS,check=False)
    except subprocess.TimeoutExpired:
        log.warning("timeout after %ss for %s", TIME_LIMIT_SECS, t.path)
    except Exception as e:
        log.error("error tracing %s: %s", t.path, e)

def main() -> None:
    log.info("scanning %s", BIN_DIR)
    targets = build_targets(list_directories(BIN_DIR))
    if not targets:
        log.info("no ELF executables found under %s", BIN_DIR)
        return

    for t in targets:
        log.info("tracing %s -> %s", t.path, t.log_path)
        run_strace(t)

    log.info("done. logs are in: %s", OUTPUT_DIR)

if __name__ == "__main__":
    main()

In [ ]:
!ls Linux-Malware-Samples/

In [ ]:
!chmod +x Linux-Malware-Samples/*

In [ ]:
import csv
import re
from pathlib import Path
from collections import Counter
from typing import Dict, List, Optional, Tuple

LOG_DIR = Path("malware_strace_logs")
LOG_GLOB = "*.strace.log"
OUT_CSV = Path("./malware_features_20.csv")

# for matching five groups  in this regex are used
LINE_RE = re.compile(r"""
    ^\s*
    (\d+)\s+                      # PID
    (\d+\.\d+)\s+                 # epoch timestamp with fraction
    ([a-zA-Z0-9_]+)               # syscall name
    \((.*)\)                      # args (greedy until last ')')
    (?:\s*=\s*([^\n]+))?          # optional result part to end
    \s*$
""", re.X)

def parse_line(line: str) -> Optional[Tuple[int, float, str, str, str]]:
    # Skip strace unwanted lines
    if line.startswith("+++ ") or line.startswith("--- SIG") or "<unfinished" in line or "resumed>" in line:
        return None
    m = LINE_RE.match(line)
    if not m:
        return None
    pid_s, ts_s, name, args, result = m.groups()
    return int(pid_s), float(ts_s), name, args or "", (result or "").strip()

ORDERED_COLUMNS = [
    "io_rw",
    "io_meta",
    "net_total",
    "net_ipv4",
    "net_ipv6",
    "net_unix",
    "mem_mgmt",
    "process_create",
    "execve",
    "priv_ctrl",
    "errno_total",
    "errno_perm",
    "errno_enoent",
    "touch_sensitive",
    "w_x_transition",
    "uses_unlink",
    "duration",
    "rate_cps",
    "err_rate",
    "net_ratio",
]

def featurize_20(path: Path) -> Dict[str, float]:
    feats = Counter()
    timestamps: List[float] = []
    net_calls = 0
    file_calls = 0
    err_calls = 0
    exec_calls = 0
    fork_calls = 0

    for raw in path.read_text(errors="ignore").splitlines():
        parsed = parse_line(raw)
        if not parsed:
            continue
        _pid, ts, name, args, result = parsed
        timestamps.append(ts)

        # Io read/write samples
        if name in ("read","write","pwrite64","pread64"):
            feats["io_rw"] += 1

        # File and meta ops include openat/newfstatat dominating your sample
        if name in ("open","openat","openat2","close","stat","fstat","lstat","newfstatat","fstatat","access","rename","unlink"):
            feats["io_meta"] += 1
            file_calls += 1

        # Networking (none in the provided snippet, but kept)
        if name in ("socket","connect","sendto","recvfrom","sendmsg","recvmsg","bind","listen","accept","shutdown","setsockopt"):
            feats["net_total"] += 1
            net_calls += 1

        # Memory management (mmap, brk present in your sample)
        if name in ("mmap","mprotect","munmap","brk"):
            feats["mem_mgmt"] += 1

        # Process/exec
        if name == "execve":
            feats["execve"] += 1
            exec_calls += 1
        if name in ("fork","vfork","clone"):
            feats["process_create"] += 1
            fork_calls += 1
        if name in ("ptrace","prctl","seccomp","capset","setuid","setgid"):
            feats["priv_ctrl"] += 1

        # Errors from result field like "-1 ENOENT (No such file or directory)"
        if result.startswith("-1"):
            feats["errno_total"] += 1
            err_calls += 1
            if "EACCES" in result or "EPERM" in result:
                feats["errno_perm"] += 1
            if "ENOENT" in result:
                feats["errno_enoent"] += 1

        # Sensitive paths and flags
        if '"/etc/' in args or '"/root' in args or '"/proc/' in args or '"/dev/mem' in args:
            feats["touch_sensitive"] = 1.0
        if name == "mprotect" and "PROT_EXEC" in args and "PROT_WRITE" in args:
            feats["w_x_transition"] = 1.0
        if name == "unlink":
            feats["uses_unlink"] = 1.0

        # Network families from args
        if "AF_INET6" in args or "inet6" in args:
            feats["net_ipv6"] += 1
        elif "AF_INET" in args or "inet" in args:
            feats["net_ipv4"] += 1
        elif "AF_UNIX" in args or "unix" in args:
            feats["net_unix"] += 1

    # Timing based on first/last timestamps; your sample ends with exit_group
    if len(timestamps) >= 2:
        timestamps.sort()
        duration = timestamps[-1] - timestamps[0]
        feats["duration"] = duration
        total_count = sum(v for k, v in feats.items() if k not in ("touch_sensitive","w_x_transition","uses_unlink"))
        feats["rate_cps"] = (total_count / duration) if duration > 0 else 0.0
    else:
        feats["duration"] = 0.0
        feats["rate_cps"] = 0.0

    denom_err = (err_calls + (feats["io_rw"] + feats["io_meta"] + feats["net_total"] +
                              feats["mem_mgmt"] + feats["process_create"] + feats["priv_ctrl"] +
                              feats["execve"]))
    feats["err_rate"] = (err_calls / denom_err) if denom_err > 0 else 0.0
    feats["net_ratio"] = (net_calls / (net_calls + file_calls)) if (net_calls + file_calls) > 0 else 0.0

    return {k: float(feats.get(k, 0.0)) for k in ORDERED_COLUMNS}

def gather_logs(root: Path) -> List[Path]:
    return sorted(root.glob(LOG_GLOB))

def write_csv(rows: List[Dict[str, float]], paths: List[Path], out_csv: Path):
    header = ["binary","log_path"] + ORDERED_COLUMNS
    with out_csv.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=header)
        w.writeheader()
        for feats, p in zip(rows, paths):
            row = {"binary": p.stem, "log_path": str(p)}
            row.update(feats)
            w.writerow(row)

def main():
    logs = gather_logs(LOG_DIR)
    if not logs:
        print(f"No logs found in {LOG_DIR} matching {LOG_GLOB}")
        return
    rows = [featurize_20(p) for p in logs]
    write_csv(rows, logs, OUT_CSV)
    print(f"Wrote {len(rows)} rows to {OUT_CSV}")

if __name__ == "__main__":
    main()


Wrote 402 rows to malware_features_20.csv


In [ ]:
!ls

event_logs  features_20.csv  sample_data  strace_logs  typescript


In [ ]:
#!/usr/bin/env python3
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from joblib import dump

CSV_PATH = "features_20.csv"  # my uploaded dataset
MODEL_PATH = "iforest_strace.joblib"

# 20 numeric feature columns as in my CSV
FEATURES = [
    "io_rw","io_meta","net_total","net_ipv4","net_ipv6","net_unix",
    "mem_mgmt","process_create","execve","priv_ctrl",
    "errno_total","errno_perm","errno_enoent",
    "touch_sensitive","w_x_transition","uses_unlink",
    "duration","rate_cps","err_rate","net_ratio"
]

def main():
    df = pd.read_csv(CSV_PATH)
    X = df[FEATURES].astype(float)

    # Isolation Forest: using adjust contamination
    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("iforest", IsolationForest(
            n_estimators=300,
            max_samples="auto",
            contamination=0.05,
            random_state=42,
            n_jobs=-1,
            bootstrap=False
        )),
    ])

    pipe.fit(X)
    dump({"pipeline": pipe, "features": FEATURES}, MODEL_PATH)
    print(f"Model  saved to {MODEL_PATH}. Trained on {len(X)} rows.")

if __name__ == "__main__":
    main()


Model  saved to iforest_strace.joblib. Trained on 507 rows.


In [ ]:
#!/usr/bin/env python3
import pandas as pd
from joblib import load
from sklearn.metrics import classification_report, confusion_matrix
import sys

# Set the paths directly here.
MODEL_PATH = "iforest_strace.joblib"
MALWARE_CSV_PATH = "malware_features_20.csv"

# --- Script ---

def evaluate_model(model_path: str, data_path: str):
    """
    Loads a model and evaluates it against a labeled dataset.
    """
    # --- 1. Load Model and Data ---
    try:
        bundle = load(model_path)
        pipe = bundle["pipeline"]
        features = bundle["features"]
        print(f"Model loaded successfully from {model_path}")
    except FileNotFoundError:
        print(f"Error: Model file not found at '{model_path}'")
        return # Use return instead of sys.exit() in a notebook
    except Exception as e:
        print(f"Error loading model: {e}")
        return

    try:
        df = pd.read_csv(data_path)
        print(f"Malware data loaded successfully from {data_path}")
    except FileNotFoundError:
        print(f"Error: Data file not found at '{data_path}'")
        return
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return

    # --- 2. Validate Data ---
    required_cols = features + ['label']
    if not all(col in df.columns for col in required_cols):
        print("Error: The CSV file is missing required columns.")
        print(f"It must contain all of these: {required_cols}")
        return

    X_test = df[features].astype(float)
    y_true = df['label'].astype(int)

    # --- 3. Make Predictions ---
    print("\nMaking predictions...")
    y_pred = pipe.predict(X_test)

    # --- 4. Show Evaluation Metrics ---
    print("\n--- Evaluation Report ---")
    report = classification_report(y_true, y_pred, target_names=['malware (-1)', 'benign (1)'])
    print(report)

    print("--- Confusion Matrix ---")
    cm = confusion_matrix(y_true, y_pred)
    print("                Predicted Malware   Predicted Benign")
    print(f"Actual Malware:        {cm[0][0]:<10}      {cm[0][1]:<10}")
    print(f"Actual Benign:         {cm[1][0]:<10}      {cm[1][1]:<10}")


if __name__ == "__main__":
    # Removed the sys.argv logic. The script now ONLY uses the
    # hardcoded paths defined at the top.
    evaluate_model(MODEL_PATH, MALWARE_CSV_PATH)


Model loaded successfully from iforest_strace.joblib
Malware data loaded successfully from malware_features_20.csv
Error: The CSV file is missing required columns.
It must contain all of these: ['io_rw', 'io_meta', 'net_total', 'net_ipv4', 'net_ipv6', 'net_unix', 'mem_mgmt', 'process_create', 'execve', 'priv_ctrl', 'errno_total', 'errno_perm', 'errno_enoent', 'touch_sensitive', 'w_x_transition', 'uses_unlink', 'duration', 'rate_cps', 'err_rate', 'net_ratio', 'label']


In [ ]:
!git clone https://github.com/MalwareSamples/Linux-Malware-Samples.git

Cloning into 'Linux-Malware-Samples'...
remote: Enumerating objects: 486, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 486 (delta 1), reused 4 (delta 1), pack-reused 481 (from 1)
Receiving objects: 100% (486/486), 510.67 MiB | 19.42 MiB/s, done.
Resolving deltas: 100% (103/103), done.
Updating files: 100% (476/476), done.
